# Financial Document Parser - Notebook

This notebook parses the SEC 10-K filing into structured JSON format.

In [ ]:
import os

# Fix numpy corruption and install Colab-compatible dependencies
print("🔧 Fixing numpy installation...")

# Force reinstall numpy with correct version
!pip uninstall -y numpy
!pip install -q "numpy==1.26.4"

print("📦 Installing remaining dependencies...")

# Install core dependencies
!pip install -q opencv-python-headless==4.8.1.78
!pip install -q pdfminer.six pdf2image pypdf pypdfium2 pillow pi-heif

# Install unstructured WITHOUT forcing dependency upgrades
!pip install -q --no-deps unstructured unstructured-inference

# Install unstructured dependencies manually
!pip install -q layoutparser lxml emoji langdetect nltk tabulate python-magic-bin beautifulsoup4

# Install project dependencies
!pip install -q neo4j chromadb sentence-transformers streamlit

print("✅ Installation complete! Please restart the kernel if you see any errors.")
print("   Runtime → Restart runtime, then re-run from cell 2")

📦 Installing from requirements.txt...
✅ Installation complete!


## ⚠️ IMPORTANT: Restart Runtime After Installation

After running the installation cell above:
1. Click **Runtime → Restart runtime** in the menu
2. DO NOT click "Run all" 
3. Manually run cells starting from cell 4

This clears the corrupted numpy from memory.

## Install Required Libraries

Run this cell first to install all dependencies.

**Colab-specific notes:**
- Uses `--no-deps` for unstructured to avoid pandas version conflicts
- Installs opencv-python-headless (Colab-compatible)
- Keeps numpy <2.0 for TensorFlow compatibility
- Upload your `src` folder before running cell 4

In [22]:
# Import required modules
import sys
import os
import json

# Add src directory to path so we can import parser
src_path = os.path.join(os.getcwd(), 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# Import the parser module
import parser as doc_parser
process_document = doc_parser.process_document

In [ ]:
# Verify critical packages
import sys

packages_to_check = {
    'cv2': 'opencv-python',
    'numpy': 'numpy',
    'pandas': 'pandas',
    'unstructured': 'unstructured',
    'pdfminer': 'pdfminer.six',
    'neo4j': 'neo4j',
    'chromadb': 'chromadb',
    'sentence_transformers': 'sentence-transformers'
}

print("🔍 Checking installed packages...\n")
all_ok = True

for module, package in packages_to_check.items():
    try:
        mod = __import__(module)
        version = getattr(mod, '__version__', 'unknown')
        print(f"✅ {package:25} v{version}")
    except ImportError:
        print(f"❌ {package:25} NOT INSTALLED")
        all_ok = False

if all_ok:
    print("\n✅ All packages installed successfully!")
else:
    print("\n⚠️ Some packages are missing - re-run cell 2")

## Verify Installation

Check that key packages are installed correctly.

## Set File Paths

Define the input PDF and output JSON paths.

In [23]:
# Define paths
BASE_DIR = os.getcwd()
pdf_path = os.path.join(BASE_DIR, "SEC 10-K Filing.pdf")
output_path = os.path.join(BASE_DIR, "parsed_data.json")

# Verify PDF exists
if os.path.exists(pdf_path):
    print(f"✅ PDF found: {pdf_path}")
    print(f"📄 File size: {os.path.getsize(pdf_path) / 1024 / 1024:.2f} MB")
else:
    print(f"❌ PDF not found at: {pdf_path}")

✅ PDF found: /content/SEC 10-K Filing.pdf
📄 File size: 8.85 MB


In [29]:
# Check GPU availability
try:
    import torch
    
    if torch.cuda.is_available():
        print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
        print(f"💾 GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
        print(f"📊 Memory allocated: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")
    else:
        print("❌ No GPU detected - running on CPU")
        print("💡 Enable GPU: Runtime → Change runtime type → GPU")
except ImportError:
    print("⚠️ PyTorch not installed - GPU check unavailable")
    print("💡 Install with: !pip install torch")

✅ GPU available: Tesla T4
💾 GPU Memory: 15.64 GB
📊 Memory allocated: 0.00 GB


## Check GPU Status

Verify GPU is available and being detected.

## Run Parser

Choose your parsing method:
- `method="unstructured"` with `strategy="fast"` → ~1-2 minutes, extracts tables
- `method="pdfminer"` → ~5-15 seconds, plain text only

In [28]:
# Run the parser
process_document(
    pdf_path=pdf_path,
    output_path=output_path,
    method="unstructured",     # or "pdfminer" for fastest parsing
    strategy="fast",            # "fast" = 1-2 min, "hi_res" = 3-5 min with GPU (but can hang)
    use_tables=True             # Extract tables as HTML
)
print("✅ Parsing complete!")

Looking for PDF at: /content/SEC 10-K Filing.pdf
Parsing document using [UNSTRUCTURED] method...
Using unstructured library for extraction...


KeyboardInterrupt: 

## Inspect Results

Load and display summary statistics of the parsed data.

In [26]:
# Load the parsed data
with open(output_path, 'r', encoding='utf-8') as f:
    parsed_data = json.load(f)

# Display summary
print(f"📊 Total sections parsed: {len(parsed_data)}")
print(f"📝 Total words: {sum(s.get('word_count', 0) for s in parsed_data):,}")
print(f"🔗 Sections with references: {len([s for s in parsed_data if s['references']])}")
print(f"📋 Sections with tables: {len([s for s in parsed_data if s.get('has_tables')])}")

📊 Total sections parsed: 0
📝 Total words: 0
🔗 Sections with references: 0
📋 Sections with tables: 0


## List All Sections

Show all section IDs with their parts and references.

In [27]:
# Display all sections
print("\n📑 All Sections:\n" + "="*80)
for i, section in enumerate(parsed_data, 1):
    part = section.get('part') or '—'
    refs = f" → {', '.join(section['references'])}" if section['references'] else ""
    words = section.get('word_count', 0)
    print(f"{i:2}. [{part:8}] {section['id']:20} ({words:,} words){refs}")


📑 All Sections:


## Search for Specific Section

Find and display a specific section by ID.

In [ ]:
# Search for a specific section
search_id = "ITEM 1A"  # Change this to search for different sections

matching_sections = [s for s in parsed_data if search_id in s['id']]

if matching_sections:
    section = matching_sections[0]
    print(f"Found: {section['id']}")
    print(f"Part: {section.get('part', 'N/A')}")
    print(f"Type: {section['type']}")
    print(f"Word count: {section.get('word_count', 0):,}")
    print(f"Has tables: {section.get('has_tables', False)}")
    print(f"References: {section['references']}")
    print(f"\nFirst 500 characters of text:")
    print("-" * 80)
    print(section['text'][:500] + "...")
else:
    print(f"❌ Section '{search_id}' not found")

In [ ]:
# Download parsed data to local machine
try:
    from google.colab import files
    
    # Download the parsed JSON file
    if os.path.exists(output_path):
        print(f"📥 Downloading {output_path}...")
        files.download(output_path)
        print("✅ Download complete!")
    else:
        print(f"❌ File not found: {output_path}")
        print("Make sure you've run the parser first (cell 8)")
        
except ImportError:
    print("⚠️ Not running in Google Colab")
    print(f"💡 File location: {output_path}")
    print(f"📂 You can find the file at: {os.path.abspath(output_path)}")

## Download Parsed Data

Download the parsed JSON file to your local machine (Colab only).